# Exploración y limpieza de datos municipales

## Configuración del entorno

Importamos librerías y dependencias.

In [1]:
import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path
from IPython.display import display
from shapely.validation import explain_validity
from shapely.ops import unary_union

Configuración de rutas.

In [2]:
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
RAW_FILE = RAW_DIR / "lugares_datos_utiles.geojson"

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_FILE = PROCESSED_DIR / "lugares_datos_utiles_limpio.geojson"

## Descarga de datos

Descargar el dataset inicial desde el servicio web integral de lugares y datos útiles de la Municipalidad de Rosario: https://datosabiertos.rosario.gob.ar/dataset/cd987077-362c-44c9-bda1-043e2e4d505d

In [3]:
url = "http://ws.rosario.gov.ar/ubicaciones/public/geojson/lugaresDatosUtiles/all/true/0/all"

response = requests.get(url, timeout=60)

# Lanza una excepción si la API devuelve un error HTTP
response.raise_for_status()

print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"Tamaño de la respuesta: {len(response.content):,} bytes")

Status: 200
Content-Type: application/json
Tamaño de la respuesta: 1,940,270 bytes


In [4]:
with open(RAW_FILE, "wb") as f:
    f.write(response.content)

print(f"Archivo guardado exitosamente en: {RAW_FILE.resolve()}")

Archivo guardado exitosamente en: D:\Documentos\MaxucaS\rosario-15-minute-city\data\raw\lugares_datos_utiles.geojson


Cargar el GeoJSON en GeoPandas con el crs original.

In [5]:
gdf = gpd.read_file(RAW_FILE)

if not gdf.crs:
    gdf.set_crs("EPSG:22185", allow_override=True, inplace=True)

## Exploración inicial

### Inspección general

Veamos las primeras filas:

In [6]:
display(gdf.head())

,calculo_coord_ref,descripcion,piso,estado,calle,direccion,codigo_calle,divs_admin,letra,interseccion,...,depto,subtipo,contactos,atencion,id,codigo_interseccion,coordenada_ref,lineas_tup,titular,geometry
0,geometria,,,Publicado,1332,1332 3177,5150.0,"[{'codigo': '1', 'nombreAbrev': 'NORTE', 'tipo...",NaN,NaN,...,,LUGAR,[],,5914,NaN,"[5432148.768664516, 6361708.56602769]","[102 Roja, 107 Negra, 107 Roja, 153 Negra]",Municipal,POINT (5432148.769 6361708.566)
1,geometria,,,Publicado,CARRASCO EUDORO,CARRASCO EUDORO 3650,36600.0,"[{'codigo': '1', 'nombreAbrev': 'NORTE', 'tipo...",NaN,NaN,...,,LUGAR,"[{'tipo': 'Teléfono', 'descripcion': '15402101...",,1687,NaN,"[5435654.73, 6362296.91]",[153 Negra],Privado,POINT (5435654.73 6362296.91)
2,geometria,,,Publicado,CORRIENTES,CORRIENTES 2879,43250.0,"[{'codigo': '3', 'nombreAbrev': 'CENTRO', 'tip...",NaN,NaN,...,,LUGAR,[],,5670,NaN,"[5439390.382258602, 6352349.181511768]","[103 Negra, 103 Roja, 107 Negra, 107 Roja, 110...",Municipal,POINT (5439390.382 6352349.182)
3,geometria,,,Publicado,CORRIENTES,CORRIENTES 2114,43250.0,"[{'codigo': '3', 'nombreAbrev': 'CENTRO', 'tip...",NaN,NaN,...,,LUGAR,[],,5669,NaN,"[5439562.35510406, 6353331.61534218]","[103 Negra, 103 Roja, 107 Negra, 107 Roja, 116...",Municipal,POINT (5439562.355 6353331.615)
4,geometria,,,Publicado,SANTA FE,SANTA FE 638,86550.0,"[{'codigo': '3', 'nombreAbrev': 'CENTRO', 'tip...",NaN,NaN,...,,LUGAR,[],,5668,NaN,"[5440940.00234185, 6354922.38402761]","[101 Negra, 102 Roja, 106 Ibarlucea, 106 Munic...",Municipal,POINT (5440940.002 6354922.384)


Su estructura:

In [7]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 981 entries, 0 to 980
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   calculo_coord_ref     981 non-null    str     
 1   descripcion           981 non-null    str     
 2   piso                  707 non-null    str     
 3   estado                981 non-null    str     
 4   calle                 981 non-null    str     
 5   direccion             981 non-null    str     
 6   codigo_calle          912 non-null    float64 
 7   divs_admin            981 non-null    object  
 8   letra                 3 non-null      str     
 9   interseccion          8 non-null      str     
 10  tipo_geometria        981 non-null    str     
 11  bis                   981 non-null    bool    
 12  ultima_actualizacion  981 non-null    str     
 13  multimedia            981 non-null    object  
 14  altura                904 non-null    float64 
 15

### Geometrías

Exploremos la columna **geometry**.

In [8]:
display(gdf.geometry.geom_type.value_counts(dropna=False))

Point           897
MultiPolygon     79
Polygon           3
LineString        2
Name: count, dtype: int64

In [9]:
print("Geometrías nulas:", gdf.geometry.isna().sum())
print("Geometrías vacías:", gdf.geometry.is_empty.sum())
print("Geometrías inválidas:", (~gdf.geometry.is_valid & gdf.geometry.notna()).sum())

Geometrías nulas: 0
Geometrías vacías: 0
Geometrías inválidas: 2


Encontremos cuáles son esas geometrías inválidas y por qué:

In [10]:
# Filtrar el GeoDataFrame
gdf_geometrias_invalidas = gdf[~gdf.geometry.is_valid & gdf.geometry.notna()]
# Usar Shapely para ver exactamente cuál es el error geométrico
for index, row in gdf_geometrias_invalidas.iterrows():
    motivo = explain_validity(row.geometry)
    print(f"ID {row['id']} ({row['name']}): {motivo}")

ID 5438 (Bosque de los constituyentes): Self-intersection[5432656.24000005 6359777.33000007]
ID 5440 (Parque Alem): Self-intersection[5436350.52000006 6359056.33000007]


Reparar geometrías

In [11]:
gdf['geometry'] = gdf.geometry.make_valid()
print("Geometrías inválidas restantes:", (~gdf.geometry.is_valid & gdf.geometry.notna()).sum())

Geometrías inválidas restantes: 0


In [12]:
display(gdf.geometry.geom_type.value_counts(dropna=False))

Point                 897
MultiPolygon           77
Polygon                 3
LineString              2
GeometryCollection      2
Name: count, dtype: int64

Como podemos observar, se convirtieron en GeometryCollection, para algunas operaciones de análisis espacial este tipo de geometría puede darnos errores por lo que procederemos a limpiarlas extrayendo los polígonos que hayan en cada colección.

In [13]:
gdf['geometry'] = gdf['geometry'].apply(
    lambda geom: unary_union([g for g in geom.geoms if g.geom_type in ['Polygon', 'MultiPolygon']]) 
    if geom.geom_type == 'GeometryCollection' else geom
)

In [14]:
display(gdf.geometry.geom_type.value_counts(dropna=False))

Point           897
MultiPolygon     79
Polygon           3
LineString        2
Name: count, dtype: int64

Revisamos que el dataset esté dentro de los límites de la ciudad.

In [15]:
minx, miny, maxx, maxy = gdf.total_bounds
print(f"X: {minx:.5f} a {maxx:.5f} metros")
print(f"Y: {miny:.5f} a {maxy:.5f} metros")

X: 5427644.27000 a 5442573.35722 metros
Y: 6345899.42000 a 6363171.71000 metros


### Etiquetas

Revisemos la columna **etiquetas**.

In [16]:
display(gdf[['name', 'etiquetas']].head())

,name,etiquetas
0,Centro de Recepción de Residuos Reciclables,"[{'id': 77, 'nombre': 'Datos útiles'}]"
1,"Embarcadero ""Costa Alta""","[{'id': 77, 'nombre': 'Datos útiles'}]"
2,Dirección Provincial de Políticas de Género,"[{'id': 77, 'nombre': 'Datos útiles'}]"
3,Centro de Asistencia Judicial,"[{'id': 77, 'nombre': 'Datos útiles'}]"
4,Centro de Protección Alicia Moreau,"[{'id': 77, 'nombre': 'Datos útiles'}]"


Notamos que los valores de esta columna son listas de Python con un diccionario de claves {'id', 'nombre'} para cada etiqueta que tenga el registro. Procederemos a simplificar esta estructura quedándonos solo con el nombre de cada etiqueta.

In [17]:
gdf['etiquetas_nombres'] = gdf['etiquetas'].apply(lambda lista: [diccionario['nombre'] for diccionario in lista] if isinstance(lista, list) else [])

Explotamos y contamos para ver todo lo que hay:

In [18]:
display(gdf.explode('etiquetas_nombres')['etiquetas_nombres'].value_counts())

etiquetas_nombres
Datos útiles              981
Oficina                   449
Realiza Trámites          296
Punto cuidar              183
Institución               116
Vecinales                  88
Espacios Verdes            82
Plaza                      72
Salud                      68
Cultura                    53
Seguridad                  48
Deporte                    41
Lugares de entrega         41
Centros Cuidar             40
Jóvenes                    36
Servicios                  33
Centro Cultural            21
Educación                  20
Bancos                     19
Pagos                      19
Turismo                    18
MuniBot                    15
Mercados y ferias          14
Parque                     12
Biblioteca                 12
Playón                     12
Justicia                   10
Ente Descentralizado        8
Museo                       8
Capacitación                6
CMD                         6
Escuelas culturales         6
Salud animal          

Decidimos agrupar etiquetas en categorías.

In [19]:
categorias = {
    'Salud': ['Salud'],
    'Educación': ['Educación', 'Escuelas culturales', 'Capacitación', 'Educativo sin nivel', 'Escuela', 'Biblioteca', 'Bibliotecas culturales'],
    'Espacios Verdes': ['Espacios Verdes', 'Plaza', 'Parque'],
    'Espacios Culturales': ['Cultura', 'Deporte', 'Centro Cultural', 'Playón', 'Museo', 'Teatro', 'Cine', 'Paseo', 'Jóvenes', 'Patrimonio', 'Turismo'],
    'Mercados y Comercios': ['Mercados y ferias', 'Centros Comerciales', 'Economía Social'],
    'Comunidad': ['Centros Cuidar', 'Punto cuidar'],
    'Servicios y Trámites': ['Bancos', 'Pagos', 'CMD', 'Vecinales', 'Justicia', 'Ente Descentralizado', 'Servicios', 'Salud animal'],
    'Seguridad': ['Seguridad']
    # Ignoramos: Datos útiles, Oficina, Realiza Trámites, Institución, Personas mayores, MuniBot, Lugares de entrega.
}

prioridad = [
    'Salud',                 
    'Educación',             
    'Espacios Verdes', 
    'Comunidad',
    'Seguridad',
    'Mercados y Comercios',  
    'Espacios Culturales',
    'Servicios y Trámites'   
]

In [20]:
gdf['categoria'] = gdf['etiquetas_nombres'].apply(
    lambda etiquetas: next((c for c in prioridad if any(e in categorias[c] for e in etiquetas)), 'Otros')
)

In [21]:
display(gdf['categoria'].value_counts())

categoria
Otros                   457
Servicios y Trámites    151
Comunidad               112
Espacios Verdes          83
Salud                    68
Seguridad                48
Educación                33
Espacios Culturales      19
Mercados y Comercios     10
Name: count, dtype: int64

### Última actualización

Veamos la columna **ultima_actualización**, que contiene la fecha en la que fue actualizado por última vez cada registro en formato "DD/MM/AAAA HH:MM". A partir de esta, crearemos una columna auxiliar transformándola a **datetime**.

In [22]:
gdf['ultima_actualizacion_dt'] = pd.to_datetime(gdf["ultima_actualizacion"], errors="coerce", dayfirst=True)

In [23]:
print("Fechas no interpretadas:", gdf['ultima_actualizacion_dt'].isna().sum())
print("Fecha más antigua:", gdf['ultima_actualizacion_dt'].min())
print("Fecha más reciente:", gdf['ultima_actualizacion_dt'].max())

Fechas no interpretadas: 0
Fecha más antigua: 2017-08-01 09:35:00
Fecha más reciente: 2026-08-27 15:18:00


In [24]:
display(gdf['ultima_actualizacion_dt'].dt.year.value_counts(dropna=False).sort_index())

ultima_actualizacion_dt
2017    307
2018     90
2019    174
2020     35
2021    129
2022     65
2023     44
2024     18
2025     82
2026     37
Name: count, dtype: int64

### Titular

In [25]:
display(gdf['titular'].value_counts(dropna=False).head(30))

titular
Municipal     758
Provincial     96
ONG            85
Privado        30
Nacional        9
Mixto           3
Name: count, dtype: int64

### Distritos y barrios

In [26]:
gdf['distrito'] = gdf['divs_admin'].apply(
    lambda lista: next((d['valor'] for d in lista if d.get('nombreTipo') == 'Distrito'), None) if isinstance(lista, list) else None
)
gdf['barrio'] = gdf['divs_admin'].apply(
    lambda lista: next((d['valor'] for d in lista if d.get('nombreTipo') == 'Barrio'), None) if isinstance(lista, list) else None
)

In [27]:
display(gdf['distrito'].value_counts(dropna=False))

distrito
CENTRO      516
NORTE       122
OESTE       102
NOROESTE     88
SUR          81
SUDOESTE     72
Name: count, dtype: int64

In [28]:
display(gdf['barrio'].value_counts(dropna=False))

barrio
CENTRO                        417
LARREA Y EMPALME GRANEROS      47
NTRA SRA DE LOURDES            40
LAS MALVINAS                   37
ALBERDI                        36
CINCO ESQUINAS                 35
ALVEAR                         33
GENERAL LAS HERAS              29
MERCEDES DE SAN MARTIN         22
TRIANGULO Y MODERNO            20
GODOY                          19
ALBERTO OLMEDO                 15
GENERAL JOSE DE SAN MARTIN     14
LUIS AGOTE                     14
URQUIZA                        14
ANTARTIDA ARGENTINA            13
ECHESORTU                      13
LA CERAMICA Y CUYO             12
ROQUE SAENZ PEÑA               12
SARMIENTO                      11
REPUBLICA DE LA 6TA            10
REPUBLICA DEL SALADILLO        10
BELGRANO                       10
LUDUEÑA SUR Y NORTE            10
LAS FLORES                     10
FISHERTON                       9
ESPAÑA Y HOSPITALES             8
DEL ABASTO                      8
LISANDRO DE LA TORRE            8
LA FLOR

### Detección de duplicados

In [29]:
if "id" in gdf.columns:
    print("IDs duplicados:", gdf["id"].duplicated().sum())

IDs duplicados: 0


In [30]:
columnas_clave = ["name", "direccion"]
duplicados = gdf.duplicated(subset=columnas_clave, keep=False)
print("Posibles duplicados por nombre y/o dirección:", duplicados.sum())

Posibles duplicados por nombre y/o dirección: 8


In [31]:
display(gdf.loc[duplicados].sort_values(columnas_clave).head())

,calculo_coord_ref,descripcion,piso,estado,calle,direccion,codigo_calle,divs_admin,letra,interseccion,...,codigo_interseccion,coordenada_ref,lineas_tup,titular,geometry,etiquetas_nombres,categoria,ultima_actualizacion_dt,distrito,barrio
85,geometria,,,Publicado,TUCUMAN,TUCUMAN 1853,92300.0,"[{'codigo': '3', 'nombreAbrev': 'CENTRO', 'tip...",NaN,NaN,...,NaN,"[5439451.52, 6355680.22]","[102 Roja, 103 Negra, 103 Roja, 106 Ibarlucea,...",Provincial,POINT (5439451.52 6355680.22),"[Ente Descentralizado, Datos útiles]",Servicios y Trámites,2019-08-28 12:49:00,CENTRO,CENTRO
520,geometria,,NaN,Publicado,TUCUMAN,TUCUMAN 1853,92300.0,"[{'codigo': '3', 'nombreAbrev': 'CENTRO', 'tip...",NaN,NaN,...,NaN,"[5439451.52, 6355680.22]","[102 Roja, 103 Negra, 103 Roja, 106 Ibarlucea,...",Provincial,POINT (5439451.52 6355680.22),"[Oficina, Realiza Trámites, Datos útiles]",Otros,2017-08-01 09:35:00,CENTRO,CENTRO
864,geometria,,,Publicado,PROVINCIAS UNIDAS,PROVINCIAS UNIDAS 150 BIS,79900.0,"[{'codigo': '2', 'nombreAbrev': 'NOROESTE', 't...",NaN,NaN,...,NaN,"[5433320.16, 6356946.53]","[101 Negra, 112 Negra, 112 Roja, 141]",Municipal,POINT (5433320.16 6356946.53),"[Ente Descentralizado, Realiza Trámites, Datos...",Servicios y Trámites,2021-08-25 14:52:00,NOROESTE,LARREA Y EMPALME GRANEROS
926,geometria,,,Publicado,PROVINCIAS UNIDAS,PROVINCIAS UNIDAS 150 BIS,79900.0,"[{'codigo': '2', 'nombreAbrev': 'NOROESTE', 't...",NaN,NaN,...,NaN,"[5433320.16, 6356946.53]","[101 Negra, 112 Negra, 112 Roja, 141]",Municipal,POINT (5433320.16 6356946.53),"[Ente Descentralizado, Oficina, Realiza Trámit...",Comunidad,2024-09-03 11:57:00,NOROESTE,LARREA Y EMPALME GRANEROS
247,geometria,,,Publicado,MENDOZA,MENDOZA 1085,67550.0,"[{'codigo': '3', 'nombreAbrev': 'CENTRO', 'tip...",NaN,NaN,...,NaN,"[5440225.5, 6354399.32]","[102 Roja, 120, 121, 123, 126 Negra, 126 Roja,...",Provincial,POINT (5440225.5 6354399.32),"[Cultura, Datos útiles]",Espacios Culturales,2018-09-10 08:42:00,CENTRO,CENTRO


In [32]:
# Ordenar por fecha de actualización
gdf = gdf.sort_values(by='ultima_actualizacion_dt', ascending=False)
# Borrar duplicados
gdf = gdf.drop_duplicates(subset=columnas_clave, keep='first')
# Ordenar por ID
gdf = gdf.sort_values('id').reset_index(drop=True)

print(f"Cantidad de registros después de eliminar duplicados: {len(gdf)}")

Cantidad de registros después de eliminar duplicados: 977


In [33]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 977 entries, 0 to 976
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   calculo_coord_ref        977 non-null    str           
 1   descripcion              977 non-null    str           
 2   piso                     704 non-null    str           
 3   estado                   977 non-null    str           
 4   calle                    977 non-null    str           
 5   direccion                977 non-null    str           
 6   codigo_calle             908 non-null    float64       
 7   divs_admin               977 non-null    object        
 8   letra                    3 non-null      str           
 9   interseccion             8 non-null      str           
 10  tipo_geometria           977 non-null    str           
 11  bis                      977 non-null    bool          
 12  ultima_actualizacion     977

### Dataset final

Seleccionamos únicamente las columnas relevantes para el análisis espacial y creamos un GeoDataFrame consolidado.

In [34]:
processed_gdf = gdf[['id', 'name', 'direccion', 'distrito', 'barrio', 'titular', 'categoria', 'etiquetas_nombres', 'ultima_actualizacion_dt', 'geometry']].copy()

In [35]:
# Filtramos los lugares que cayeron en otros
otros = processed_gdf[processed_gdf['categoria'] == 'Otros']
print(f"Total de lugares que van a quedar afuera: {len(otros)}")

Total de lugares que van a quedar afuera: 456


In [36]:
display(otros[['name', 'etiquetas_nombres']].head(20))

,name,etiquetas_nombres
0,Secretaría de Cultura y Educación,"[Oficina, Realiza Trámites, Datos útiles]"
1,Dirección de Cartografía,"[Oficina, Datos útiles]"
2,Dirección de Gravámenes al Comercio e Industria,"[Oficina, Datos útiles]"
3,Dirección de Gravámenes Especiales,"[Oficina, Realiza Trámites, Datos útiles]"
4,Dirección de Gravámenes sobre Inmuebles,"[Oficina, Datos útiles]"
5,Dirección de Topografía,"[Oficina, Datos útiles]"
6,Dirección General de Gestión de Recursos,"[Oficina, Realiza Trámites, Datos útiles]"
7,Dirección General de Obras Particulares,"[Oficina, Realiza Trámites, Datos útiles]"
8,Dirección General de Topografía y Catastro,"[Oficina, Realiza Trámites, Datos útiles]"
9,Edificio Aduana,"[Oficina, Realiza Trámites, Datos útiles]"


In [37]:
processed_gdf = processed_gdf[processed_gdf['categoria'] != 'Otro'].reset_index(drop=True)

In [38]:
processed_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 977 entries, 0 to 976
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id                       977 non-null    int32         
 1   name                     977 non-null    str           
 2   direccion                977 non-null    str           
 3   distrito                 977 non-null    str           
 4   barrio                   977 non-null    str           
 5   titular                  977 non-null    str           
 6   categoria                977 non-null    str           
 7   etiquetas_nombres        977 non-null    object        
 8   ultima_actualizacion_dt  977 non-null    datetime64[us]
 9   geometry                 977 non-null    geometry      
dtypes: datetime64[us](1), geometry(1), int32(1), object(1), str(6)
memory usage: 72.6+ KB


In [39]:
processed_gdf.to_file(PROCESSED_FILE, driver='GeoJSON')
print(f"Archivo guardado exitosamente en: {PROCESSED_DIR.resolve()}")

Archivo guardado exitosamente en: D:\Documentos\MaxucaS\rosario-15-minute-city\data\processed
